# Zajęcie 5. Bezpieczeństwo danych

## Szyfrowanie danych (Fernet, AES)

In [1]:
from cryptography.fernet import Fernet
import pandas as pd

key = Fernet.generate_key()
cipher = Fernet(key)

df = pd.read_csv("dane_zaj1/dane_pacjentow_demo.csv")
df["diagnosis_enc"] = df["visit_date"].apply(
    lambda x: cipher.encrypt(x.encode()).decode()
)

FileNotFoundError: [Errno 2] No such file or directory: 'dane_zaj1/dane_pacjentow_demo.csv'

## Haszowanie (SHA-256)

In [ ]:
from cryptography.fernet import Fernet
import pandas as pd

key = Fernet.generate_key()
cipher = Fernet(key)

df = pd.read_csv("dane_zaj1/dane_pacjentow_demo.csv")
df["diagnosis_enc"] = df["visit_date"].apply(
    lambda x: cipher.encrypt(x.encode()).decode()
)

## Pseudonimizacja kolumny

In [ ]:
import uuid
df["patient_id"] = [str(uuid.uuid4()) for _ in range(len(df))]

# ============================================
# Bezpieczeństwo danych medycznych – Etapy 2–5
# ============================================

In [ ]:
# IMPORTY
import os
import uuid
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd

from cryptography.fernet import Fernet
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [ ]:
# -------------------------------------------------
# Przygotowanie danych (skrót Etapu 1 – dla kontekstu)
# -------------------------------------------------
PATH = "dane_pacjentow_demo.csv"

if os.path.exists(PATH):
    df = pd.read_csv(PATH)
else:
    # Demo: generujemy prosty zbiór danych, jeśli plik nie istnieje
    rng = np.random.default_rng(7)
    N = 200
    age = rng.integers(18, 90, size=N)
    sex = rng.choice(["F", "M"], size=N)
    height = rng.normal(170, 10, size=N).clip(140, 200)
    weight = rng.normal(75, 15, size=N).clip(40, 160)
    bmi = weight / (height/100)**2
    sbp = (100 + 0.5*age + 1.2*bmi + (sex=="M")*5 + rng.normal(0,10,N)).round()
    dbp = (60 + 0.2*age + 0.6*bmi + (sex=="M")*3 + rng.normal(0,6,N)).round()
    hypertension = ((sbp >= 140) | (dbp >= 90)).astype(int)

    df = pd.DataFrame({
        "age": age,
        "sex": sex,
        "bmi": bmi.round(1),
        "systolic_bp": sbp.astype(int),
        "diastolic_bp": dbp.astype(int),
        "hypertension": hypertension
    })
    df.to_csv(PATH, index=False)

print("Podgląd danych:")
print(df.head())

In [ ]:
# ============================================
# ETAP 2: SZYFROWANIE I INTEGRALNOŚĆ DANYCH
# ============================================

# --- 2.1 Szyfrowanie wybranych kolumn (np. diagnoza / etykieta) ---

# Dla przykładu tworzymy kolumnę 'diagnosis' na podstawie 'hypertension'
if "diagnosis" not in df.columns:
    df["diagnosis"] = df["hypertension"].map({1: "Hypertension", 0: "Normal"})

# Generujemy klucz symetryczny (Fernet oparty na AES)
key = Fernet.generate_key()
cipher = Fernet(key)
print("\n[Etap 2.1] Klucz szyfrujący wygenerowany.")

# Szyfrujemy kolumnę 'diagnosis' do nowej kolumny 'diagnosis_enc'
df["diagnosis_enc"] = df["diagnosis"].apply(
    lambda x: cipher.encrypt(x.encode("utf-8")).decode("utf-8")
)
print("\n[Etap 2.1] Przykład zaszyfrowanej diagnozy:")
print(df[["diagnosis", "diagnosis_enc"]].head())

# --- 2.2 Hashowanie pliku z danymi (integralność) ---

def compute_file_hash(path: str, algo: str = "sha256") -> str:
    """Zwraca skrót kryptograficzny pliku."""
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            h.update(chunk)
    return h.hexdigest()

# Zapisujemy aktualny plik i liczymy hash
df.to_csv(PATH, index=False)
original_hash = compute_file_hash(PATH, "sha256")
print("\n[Etap 2.2] SHA-256 pliku danych:")
print(original_hash)

In [ ]:
# ============================================
# ETAP 3: PSEUDONIMIZACJA I ANONIMIZACJA
# ============================================

# --- 3.1 Pseudonimizacja: sztuczny identyfikator pacjenta ---

if "patient_id" not in df.columns:
    df["patient_id"] = [str(uuid.uuid4()) for _ in range(len(df))]
print("\n[Etap 3.1] Pseudonimizacja – przykładowe identyfikatory:")
print(df[["patient_id"]].head())

# --- 3.2 Anonimizacja: tworzymy grupy wiekowe i usuwamy potencjalne identyfikatory ---

def age_to_group(age):
    if age < 30:
        return "<30"
    elif age < 50:
        return "30-49"
    elif age < 70:
        return "50-69"
    else:
        return ">=70"

df["age_group"] = df["age"].apply(age_to_group)

# Załóżmy, że 'age' to potencjalnie zbyt szczegółowa cecha – usuwamy ją po generalizacji
df_anonym = df.drop(columns=["age"])  # w realnym systemie usunęlibyśmy też imię/nazwisko, PESEL itd.

print("\n[Etap 3.2] Przykład anonimizacji (grupy wiekowe zamiast surowej wartości):")
print(df_anonym[["age_group", "bmi", "systolic_bp", "diastolic_bp"]].head())

In [ ]:
# ============================================
# ETAP 4: KONTROLA DOSTĘPU I AUDYT
# ============================================

# Prosty model RBAC – Role-Based Access Control
ROLES = {
    "admin": {
        "columns": df_anonym.columns.tolist()  # pełny dostęp
    },
    "doctor": {
        # dostęp do cech klinicznych + diagnosis (ale np. bez view na zaszyfrowaną kolumnę)
        "columns": ["patient_id", "age_group", "bmi",
                    "systolic_bp", "diastolic_bp", "hypertension", "diagnosis"]
    },
    "analyst": {
        # analityk widzi tylko dane zanonimizowane bez diagnosis
        "columns": ["age_group", "bmi", "systolic_bp", "diastolic_bp", "hypertension"]
    }
}

# Prosty dziennik audytu (log w pamięci)
audit_log = []

def audit(user: str, role: str, action: str, status: str):
    """Zapisuje wpis w logu audytowym."""
    audit_log.append({
        "time": datetime.now().isoformat(timespec="seconds"),
        "user": user,
        "role": role,
        "action": action,
        "status": status
    })


def get_data_view(df_in: pd.DataFrame, role: str, user: str = "unknown") -> pd.DataFrame:
    """Zwraca widok danych zgodny z rolą użytkownika, z logowaniem audytu."""
    if role not in ROLES:
        audit(user, role, "ACCESS_DENIED_UNKNOWN_ROLE", "denied")
        raise ValueError(f"Nieznana rola: {role}")

    allowed_cols = [c for c in ROLES[role]["columns"] if c in df_in.columns]
    view = df_in[allowed_cols].copy()
    audit(user, role, f"ACCESS_GRANTED_{','.join(allowed_cols)}", "granted")
    return view

# Przykłady użycia:
print("\n[Etap 4] Przykładowe widoki danych dla różnych ról:")

df_admin_view = get_data_view(df_anonym, role="admin", user="alice_admin")
print("\nRola admin (pierwsze 3 wiersze):")
print(df_admin_view.head(3))

df_doctor_view = get_data_view(df_anonym, role="doctor", user="bob_doctor")
print("\nRola doctor (pierwsze 3 wiersze):")
print(df_doctor_view.head(3))

df_analyst_view = get_data_view(df_anonym, role="analyst", user="carol_analyst")
print("\nRola analyst (pierwsze 3 wiersze):")
print(df_analyst_view.head(3))

print("\n[Etap 4] Log audytu:")
for entry in audit_log:
    print(entry)

In [ ]:
# ============================================
# ETAP 5: BEZPIECZEŃSTWO MODELU SI
#  - weryfikacja integralności danych
#  - walidacja wejścia
#  - (prosty przykład trenowania i predykcji)
# ============================================

# --- 5.1 Definiujemy hash referencyjny danych treningowych ---

TRAIN_HASH = original_hash  # w prawdziwym systemie zapisujemy to np. w konfiguracji lub bazie

def verify_data_integrity(path: str, expected_hash: str) -> bool:
    """Sprawdza, czy plik z danymi nie został zmodyfikowany."""
    current_hash = compute_file_hash(path, "sha256")
    return current_hash == expected_hash

print("\n[Etap 5.1] Weryfikacja integralności przed trenowaniem modelu:")
print("Integralność OK?" , verify_data_integrity(PATH, TRAIN_HASH))

# --- 5.2 Walidacja wejścia dla modelu ---

ALLOWED_AGE_GROUPS = ["<30", "30-49", "50-69", ">=70"]

def validate_input_row(row: pd.Series) -> bool:
    """Prosta walidacja wejścia: zakresy, wartości dozwolone."""
    try:
        if row["age_group"] not in ALLOWED_AGE_GROUPS:
            return False
        if not (10 <= row["bmi"] <= 60):
            return False
        if not (60 <= row["systolic_bp"] <= 260):
            return False
        if not (40 <= row["diastolic_bp"] <= 150):
            return False
    except KeyError:
        return False
    return True

print("\n[Etap 5.2] Walidacja przykładowych wierszy:")
for i in range(3):
    print(i, "=>", validate_input_row(df_anonym.iloc[i]))


# --- 5.3 Przygotowanie danych do modelu (poprawiona wersja) ---

df_model = df_anonym.copy()

# Kodowanie zmiennych kategorycznych: age_group + sex
df_model = pd.get_dummies(df_model, columns=["age_group", "sex"], drop_first=True)

X = df_model.drop(columns=[
    "hypertension", 
    "diagnosis", 
    "diagnosis_enc",
    "patient_id"
], errors="ignore")

y = df_model["hypertension"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("\n[Etap 5.3] Bezpieczne trenowanie modelu")

print("Accuracy (train):", model.score(X_train, y_train))
print("Accuracy (test): ", model.score(X_test, y_test))


# --- 5.4 Bezpieczna funkcja predykcji ---

def secure_predict(model, row: pd.Series, data_path: str, expected_hash: str):
    """
    Bezpieczna inferencja:
      1. weryfikujemy integralność pliku z danymi,
      2. walidujemy wejście,
      3. wykonujemy predykcję.
    """
    # 1. integralność
    if not verify_data_integrity(data_path, expected_hash):
        raise RuntimeError("Dane zostały zmodyfikowane – przerwij predykcję!")

    # 2. walidacja wejścia
    if not validate_input_row(row):
        raise ValueError("Nieprawidłowe dane wejściowe – odrzucono wiersz.")

    # 3. przygotowanie cech (tak jak przy trenowaniu)
    row_df = row.to_frame().T
    row_df = pd.get_dummies(row_df, columns=["age_group"], drop_first=True)
    
    # upewniamy się, że kolumny są zgodne z X_train
    for col in X_train.columns:
        if col not in row_df.columns:
            row_df[col] = 0
    row_df = row_df[X_train.columns]

    proba = model.predict_proba(row_df)[0,1]
    pred = model.predict(row_df)[0]
    return pred, proba

print("\n[Etap 5.4] Bezpieczna predykcja dla wybranego pacjenta:")

sample_row = df_anonym.iloc[0]
pred, proba = secure_predict(model, sample_row, PATH, TRAIN_HASH)
print("Pacjent 0 -> predykcja:", pred, "prawdopodobieństwo HT:", round(proba, 3))
